In [1]:
# 데이터 준비
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs, make_classification, make_regression
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Helper function to create DataFrame
def create_classification_data():
    X, y = make_classification(n_samples=100, n_features=5, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    # # feature_1에 결측치를 추가 (10% 비율로)
    # missing_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
    # df.loc[missing_indices, 'feature_1'] = np.nan

    missing_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
    print("missing", missing_indices.shape)
    for row in missing_indices:
        col = np.random.randint(1,6)
        df.loc[row, f"feature_{col}"] = np.nan
    
    return df

def create_regression_data():
    X, y = make_regression(n_samples=100, n_features=5, noise=0.1, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    for column in df.columns[:-1]:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # 이상치를 추가할 인덱스를 랜덤하게 선택
        outlier_indices = np.random.choice(df.index, size=5, replace=False)
        for idx in outlier_indices:
            df.at[idx, column] = np.random.uniform(upper_bound + 1, upper_bound + 10)

    return df

def create_blobs_data():
    X, y = make_blobs(n_samples=100, n_features=5, centers=3, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    return df


# Generate datasets for each problem type
classification_df = create_classification_data()
regression_df = create_regression_data()
blobs_df = create_blobs_data()

print('=== Classification Generated Datasets ===')
print(classification_df.head())
print('\n=== regression Generated Datasets ===')
print(regression_df.head())
print('\n=== blobs Generated Datasets ===')
print(blobs_df.head())

missing (10,)
=== Classification Generated Datasets ===
   feature_1  feature_2  feature_3  feature_4  feature_5  target
0  -0.430668   0.672873  -0.724280  -0.539630  -0.651600       0
1   0.211646  -0.843897   0.534794   0.825848   0.681953       1
2   1.092675   0.409106        NaN  -0.942751  -0.981509       0
3   1.519901  -0.773361   1.998053   0.155132  -0.385314       0
4  -0.453901  -2.183473   0.244724   2.591239  -0.484234       1

=== regression Generated Datasets ===
   feature_1  feature_2  feature_3  feature_4  feature_5      target
0   0.975120  -0.677162  -0.012247  -0.897254   0.075805  -57.195760
1   0.081874  -0.485364   0.758969  -0.772825  10.375250  -46.546477
2  -1.412304  -0.908024  -0.562288  -1.012831   0.314247 -258.133440
3  -0.645120   0.361636   1.356240  -0.072010   1.003533  115.850751
4  -0.622700   0.280992  -1.952088  -0.151785   0.588317 -123.767712

=== blobs Generated Datasets ===
   feature_1  feature_2  feature_3  feature_4  feature_5  target
0 

In [9]:
print('[전처리 전] 결측치 수:', classification_df.isna().sum().sum())

classification_df['feature_1'] = classification_df['feature_1'].fillna(classification_df['feature_1'].mean())

print('[전처리 후] 결측치 수:', classification_df.isna().sum().sum())

[전처리 전] 결측치 수: 10
[전처리 후] 결측치 수: 0


In [10]:
print('이상치 처리 전 결측치 수:', regression_df['feature_3'].isna().sum())

Q1 = regression_df['feature_3'].quantile(0.25)
Q3 = regression_df['feature_3'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

regression_df.loc[(regression_df['feature_3'] < lower_bound) | (regression_df['feature_3'] > upper_bound), 'feature_3'] = None

print('이상치 처리 후 결측치 수:', regression_df['feature_3'].isna().sum())

regression_df.dropna(subset=['feature_3'], inplace=True)

print('결측치 삭제:', regression_df['feature_3'].isna().sum())

이상치 처리 전 결측치 수: 0
이상치 처리 후 결측치 수: 5
결측치 삭제: 0


In [34]:
temp = create_classification_data()

missing (10,)


In [35]:
temp.isna().sum()

feature_1    2
feature_2    1
feature_3    2
feature_4    4
feature_5    1
target       0
dtype: int64

In [37]:
temp.describe()

,feature_1,feature_2,feature_3,feature_4,feature_5,target
count,98.000000,99.000000,98.000000,96.000000,99.000000,100.000000
mean,-0.023934,0.051833,-0.066678,0.003061,0.013488,0.500000
std,0.896249,1.273653,1.234464,1.313664,0.999956,0.502519
min,-1.692005,-2.683180,-2.523434,-2.585909,-3.241267,0.000000
25%,-0.809387,-0.972043,-1.126223,-1.138573,-0.637150,0.000000
50%,0.098274,-0.078734,-0.082858,0.113184,0.045572,0.500000
75%,0.714228,1.194954,1.007276,1.089659,0.684107,1.000000
max,1.724002,2.489048,2.388694,2.591239,2.314659,1.000000


In [6]:
classification_df.groupby('target')[['feature_1', 'feature_2']].mean()

,feature_1,feature_2
target,,
0,0.148845,0.993215
1,-0.191222,-0.905930


In [11]:
pd.get_dummies(blobs_df, columns=['target'])

,feature_1,feature_2,feature_3,feature_4,feature_5,target_0,target_1,target_2
0,-3.002630,9.937449,6.346484,2.846759,-6.870483,True,False,False
1,-5.970371,-9.040785,7.384420,2.669463,4.287546,False,True,False
2,-2.290559,9.896047,3.630793,0.389875,-6.105927,True,False,False
3,-3.463695,8.263107,3.509451,2.743147,-5.611238,True,False,False
4,-1.888649,8.853349,4.251614,1.087657,-7.236372,True,False,False
...,...,...,...,...,...,...,...,...
95,-5.813911,-9.797336,8.705528,2.927422,3.557547,False,True,False
96,-2.334365,7.797431,5.689226,3.298275,-6.145126,True,False,False
97,-3.138673,9.612007,7.199367,2.367403,-6.757408,True,False,False
98,-1.927075,9.902035,5.534211,2.728167,-7.086793,True,False,False


In [12]:
blobs_df['target'].unique()

array([0, 1, 2])

In [16]:
blobs_df[['feature_1', 'feature_2']]

,feature_1,feature_2
0,-3.002630,9.937449
1,-5.970371,-9.040785
2,-2.290559,9.896047
3,-3.463695,8.263107
4,-1.888649,8.853349
...,...,...
95,-5.813911,-9.797336
96,-2.334365,7.797431
97,-3.138673,9.612007
98,-1.927075,9.902035
